# Rank-Abundance Threshold Selection

Retain vOTUs whose trimmed mean exceeds the strongest eligible microbial background.

## Notebook Setup

In [1]:
# Load core libraries and define notebook settings
from pathlib import Path
from scipy.stats import fisher_exact

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting preferences
sns.set_style(
    "ticks",
    {
        "axes.grid": True,
        "grid.color": "#BFBFBF",
        "grid.linestyle": "-",
        "grid.linewidth": 0.35,
        "axes.facecolor": "white",
        "figure.facecolor": "white",
    },
)
sns.set_palette("colorblind")

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.transparent": False,
    "savefig.bbox": "tight",
    "axes.linewidth": 0.7,
    "axes.edgecolor": "#222222",
    "xtick.major.width": 0.7,
    "ytick.major.width": 0.7,
    "xtick.minor.width": 0.5,
    "ytick.minor.width": 0.5,
    "xtick.major.size": 4.5,
    "ytick.major.size": 4.5,
    "xtick.minor.size": 2.5,
    "ytick.minor.size": 2.5,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "grid.linewidth": 0.35,
    "grid.alpha": 0.8,
    "patch.linewidth": 0.7,
    "lines.linewidth": 1.0,
    "lines.markersize": 4,
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "text.usetex": False,
    "font.size": 5,
    "axes.titlepad": 5,
    "axes.titlesize": 7,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "legend.frameon": True,
    "legend.fancybox": False,
    "legend.framealpha": 1.0,
    "legend.edgecolor": "#444444",
    "legend.facecolor": "white",
    "legend.fontsize": 5,
    "legend.title_fontsize": 6,
    "savefig.dpi": 300,
    "figure.dpi": 150,
    "svg.fonttype": "none",
})

PROJECT_DIR = Path("/home/lmf/PhylloVir")
VIRAL_WORLD_DIR = PROJECT_DIR / "VIRAL_WORLD"
VIRAL_FRACTION_DIR = VIRAL_WORLD_DIR / "VIRAL_FRACTION"
FIGURES_TABLES_DIR = VIRAL_FRACTION_DIR / "FIGURES_AND_TABLES"
SUP_FIGS_TABLES_DIR = FIGURES_TABLES_DIR / "SUP_FIGS_TABLES"
SUP_FIGS_TABLES_DIR.mkdir(parents=True, exist_ok=True)
BACTERIAL_FRACTION_DIR = VIRAL_WORLD_DIR / "BACTERIAL_FRACTION"

VIRAL_MAPPING_DIR = VIRAL_FRACTION_DIR / "06_MAPPING"
VIRAL_MAPPING_REFERENCES_DIR = VIRAL_MAPPING_DIR / "REFERENCES"
STRAINS_GENOMAD_SUMMARY_DIR = (
    BACTERIAL_FRACTION_DIR
    / "MICROBIAL_GENOMES_PHYLLOVIR"
    / "strains_in_microbial_fraction_2018_genomad"
    / "strains_in_microbial_fraction_2018_summary"
)

VIRAL_PLASMID_SUMMARY_PATH = (
    VIRAL_FRACTION_DIR
    / "04_VIRAL_ID"
    / "ALL_spades_filtered_scaffolds.tot_plasmid_summary.tsv"
)
BACTERIAL_PLASMID_SUMMARY_PATH = (
    BACTERIAL_FRACTION_DIR
    / "04_VIRAL_ID"
    / "ALL_spades_filtered_scaffolds.tot_plasmid_summary.tsv"
)
STRAIN_PLASMID_SUMMARY_PATH = (
    STRAINS_GENOMAD_SUMMARY_DIR
    / "strains_in_microbial_fraction_2018_plasmid_summary.tsv"
)
PREDICTED_PHAGE_BACKGROUND_PATH = (
    STRAINS_GENOMAD_SUMMARY_DIR
    / "strains_in_microbial_fraction_2018_virus_summary.tsv"
)
RPKM_NO_THRESHOLD_PATH = VIRAL_MAPPING_DIR / "filtered_RPKM_normalised_tot.txt"
VOTU_METADATA_PATH = VIRAL_FRACTION_DIR / "vOTU_metadata.csv"

OUTDIR = (
    VIRAL_FRACTION_DIR
    / "FIGURES_AND_TABLES"
    / "SUPPLEMENTARY"
    / "11_rank_abundance_threshold_selection"
)
OUTDIR.mkdir(parents=True, exist_ok=True)

SAMPLES = [
    "vFL", "vOL", "vF23",
    "vN1", "vN2", "vN3",
    "vZ1", "vZ2", "vZ3",
    "vF1A", "vF2A", "vF3A",
    "vF1B", "vF2B", "vF3B",
    "vF1C", "vF2C", "vF3C",
    "vO1A", "vO2A", "vO3A",
    "vO1B", "vO2B", "vO3B",
    "vO1C", "vO2C", "vO3C",
    "vS1B", "vS2B", "vS3B",
    "vS1C", "vS2C", "vS3C",
    "vNWd", "vSWd", "vSEd",
]

THRESHOLD_MULTIPLIERS = [1.0, 1.25, 1.5, 2.0, 3.0, 5.0]
DEFAULT_THRESHOLD = 1.5
MIN_BACKGROUND_LENGTH = 10_000
BACKGROUND_ORIGINS = {"MICROBIAL", "CULTURE", "MG_ASSEMBLY"}

def threshold_key(multiplier):
    return f"{multiplier:.1f}" if float(multiplier).is_integer() else str(multiplier)

THRESHOLD_KEYS = [threshold_key(value) for value in THRESHOLD_MULTIPLIERS]
DEFAULT_THRESHOLD_KEY = threshold_key(DEFAULT_THRESHOLD)


## Eligible Microbial Background

Select the strongest eligible reference independently for each sample.

In [2]:
# Define the eligible microbial background
plasmid_paths = [
    VIRAL_PLASMID_SUMMARY_PATH,
    BACTERIAL_PLASMID_SUMMARY_PATH,
    STRAIN_PLASMID_SUMMARY_PATH,
]

plasmid_tables = []
for plasmid_path in plasmid_paths:
    plasmid_table = pd.read_csv(plasmid_path, sep="\t")
    plasmid_table = plasmid_table.loc[
        plasmid_table["seq_name"].astype(str).ne("seq_name")
    ].copy()
    plasmid_tables.append(plasmid_table)

plasmids_all = pd.concat(plasmid_tables, ignore_index=True)
for column in ["length", "plasmid_score", "n_hallmarks", "marker_enrichment"]:
    plasmids_all[column] = pd.to_numeric(plasmids_all[column], errors="coerce")

conservative_plasmid_ids = set(
    plasmids_all.loc[
        plasmids_all["length"].ge(MIN_BACKGROUND_LENGTH)
        & plasmids_all["plasmid_score"].gt(0.8)
        & plasmids_all["n_hallmarks"].ge(1)
        & plasmids_all["marker_enrichment"].ge(1.5),
        "seq_name",
    ]
    .dropna()
    .astype(str)
)

predicted_phage_background_ids = set(
    pd.read_csv(
        PREDICTED_PHAGE_BACKGROUND_PATH,
        sep="\t",
        usecols=["seq_name"],
    )["seq_name"]
    .dropna()
    .astype(str)
)

rpkm_no_threshold = pd.read_csv(RPKM_NO_THRESHOLD_PATH)
votu_metadata = pd.read_csv(VOTU_METADATA_PATH, low_memory=False)
votu_metadata["vOTU"] = votu_metadata["vOTU"].astype(str)
comparison_flag_candidates = [
    "detected_in_comparison_panel",  # Current descriptive metadata schema.
    "COMPARISON",                   # Legacy schema from the original analysis notebook.
]
comparison_flag_column = next(
    (column for column in comparison_flag_candidates if column in votu_metadata.columns),
    None,
)
if comparison_flag_column is None:
    raise ValueError(
        "The vOTU metadata lacks a comparison-panel flag. Expected one of: "
        f"{comparison_flag_candidates}"
    )
comparison_panel_mask = (
    votu_metadata[comparison_flag_column]
    .astype("string")
    .str.strip()
    .str.lower()
    .isin(["true", "1", "yes"])
)
print(f"Using vOTU comparison-panel flag: {comparison_flag_column}")
rpkm_no_threshold["vOTU"] = rpkm_no_threshold["vOTU"].astype(str)

missing_samples = sorted(set(SAMPLES).difference(rpkm_no_threshold.columns))
if missing_samples:
    raise ValueError(
        "The no-threshold RPKM table is missing sample columns: "
        f"{missing_samples}"
    )

detected_votu_ids = set(
    rpkm_no_threshold.loc[
        rpkm_no_threshold[SAMPLES].gt(0).any(axis=1),
        "vOTU",
    ]
)
phyllosphere_votu_ids = set(
    votu_metadata.loc[
        comparison_panel_mask
        & votu_metadata["vOTU"].isin(detected_votu_ids),
        "vOTU",
    ]
)

background_input_summary = pd.DataFrame(
    [
        {
            "metric": "Conservative plasmid references excluded",
            "value": len(conservative_plasmid_ids),
        },
        {
            "metric": "Predicted viral strain references excluded",
            "value": len(predicted_phage_background_ids),
        },
        {
            "metric": "Detected phyllosphere vOTUs tested",
            "value": len(phyllosphere_votu_ids),
        },
    ]
)
background_input_summary


Using vOTU comparison-panel flag: detected_in_comparison_panel


,metric,value
0,Conservative plasmid references excluded,3285
1,Predicted viral strain references excluded,30212
2,Detected phyllosphere vOTUs tested,7799


## Rank-Abundance Thresholds

Evaluate the existing multiplier series and save threshold membership for notebook 14.

In [3]:
# Apply each rank-abundance threshold
covstats_columns = [
    "ID",
    "Mean",
    "Length",
    "Covered_bases",
    "Read_Count",
    "Variance",
    "Trimmed_Mean",
    "RPKM",
]

per_sample_background_rows = []
per_sample_threshold_rows = []
membership_rows = []

for sample in SAMPLES:
    microbial_path = (
        VIRAL_MAPPING_REFERENCES_DIR
        / f"bowtie2_ALL_assembled_filtered_scaffolds_MICROBIAL_10000_derreplicated.tot_{sample}_tot_covstats.txt"
    )
    viral_path = VIRAL_MAPPING_DIR / f"bowtie2_{sample}_tot_covstats.txt"
    culture_path = (
        VIRAL_MAPPING_REFERENCES_DIR
        / f"bowtie2_strains_in_microbial_fraction_2018_{sample}_tot_covstats.txt"
    )
    mg_assembly_path = (
        VIRAL_MAPPING_REFERENCES_DIR
        / f"bowtie2_ALL_MG_assembled_filtered_scaffolds_MICROBIAL_10000_derreplicated.tot_{sample}_tot_covstats.txt"
    )

    microbial = pd.read_csv(microbial_path, sep="\t")
    viral = pd.read_csv(viral_path, sep="\t")
    culture = pd.read_csv(culture_path, sep="\t")
    mg_assembly = pd.read_csv(mg_assembly_path, sep="\t")

    for table, origin in [
        (microbial, "MICROBIAL"),
        (viral, "VIRAL"),
        (culture, "CULTURE"),
        (mg_assembly, "MG_ASSEMBLY"),
    ]:
        if table.shape[1] != len(covstats_columns):
            raise ValueError(
                f"Unexpected covstats columns for {sample}, {origin}: "
                f"{table.shape[1]}"
            )
        table.columns = covstats_columns
        table["ORIGIN"] = origin

    combined = pd.concat(
        [microbial, viral, culture, mg_assembly],
        ignore_index=True,
    )
    combined["ID"] = combined["ID"].astype(str)
    combined["Trimmed_Mean"] = pd.to_numeric(
        combined["Trimmed_Mean"],
        errors="coerce",
    )
    combined["Length"] = pd.to_numeric(combined["Length"], errors="coerce")
    combined["RPKM"] = pd.to_numeric(combined["RPKM"], errors="coerce")
    combined = combined.loc[combined["Trimmed_Mean"].gt(0)].copy()
    combined.loc[
        combined["ID"].isin(conservative_plasmid_ids),
        "ORIGIN",
    ] = "PLASMID"

    eligible_background = combined.loc[
        combined["ORIGIN"].isin(BACKGROUND_ORIGINS)
        & combined["Length"].ge(MIN_BACKGROUND_LENGTH)
        & ~combined["ID"].isin(predicted_phage_background_ids)
    ].copy()
    if eligible_background.empty:
        raise ValueError(
            f"No eligible microbial background reference was found for {sample}."
        )

    top_background_index = eligible_background["Trimmed_Mean"].idxmax()
    top_background = eligible_background.loc[top_background_index]
    top_background_value = float(top_background["Trimmed_Mean"])

    per_sample_background_rows.append(
        {
            "sample": sample,
            "top_background_contig": top_background["ID"],
            "top_background_origin": top_background["ORIGIN"],
            "top_background_length": int(top_background["Length"]),
            "top_background_trimmed_mean": top_background_value,
            "top_background_rpkm": float(top_background["RPKM"]),
        }
    )

    viral_rows = combined.loc[combined["ORIGIN"].eq("VIRAL")].copy()
    for multiplier, key in zip(THRESHOLD_MULTIPLIERS, THRESHOLD_KEYS):
        cutoff = multiplier * top_background_value
        passing_ids = set(
            viral_rows.loc[
                viral_rows["Trimmed_Mean"].gt(cutoff),
                "ID",
            ]
        ).intersection(phyllosphere_votu_ids)

        per_sample_threshold_rows.append(
            {
                "sample": sample,
                "threshold_multiplier": key,
                "microbial_background_trimmed_mean": top_background_value,
                "abundance_cutoff": cutoff,
                "top_background_contig": top_background["ID"],
                "top_background_origin": top_background["ORIGIN"],
                "n_passing_phyllosphere_vOTUs": len(passing_ids),
            }
        )
        membership_rows.extend(
            {
                "threshold_multiplier": key,
                "vOTU": votu,
                "rank_abundance_call": "retained",
            }
            for votu in passing_ids
        )

per_sample_background = pd.DataFrame(per_sample_background_rows)
per_sample_threshold_summary = pd.DataFrame(per_sample_threshold_rows)
threshold_membership = (
    pd.DataFrame(
        membership_rows,
        columns=["threshold_multiplier", "vOTU", "rank_abundance_call"],
    )
    .drop_duplicates()
    .sort_values(["threshold_multiplier", "vOTU"])
    .reset_index(drop=True)
)

default_votu_ids = set(
    threshold_membership.loc[
        threshold_membership["threshold_multiplier"].eq(DEFAULT_THRESHOLD_KEY),
        "vOTU",
    ]
)
threshold_summary_rows = []
for key in THRESHOLD_KEYS:
    retained_votu_ids = set(
        threshold_membership.loc[
            threshold_membership["threshold_multiplier"].eq(key),
            "vOTU",
        ]
    )
    shared_votu_ids = retained_votu_ids.intersection(default_votu_ids)
    union_votu_ids = retained_votu_ids.union(default_votu_ids)
    threshold_summary_rows.append(
        {
            "threshold_multiplier": key,
            "n_retained_vOTUs": len(retained_votu_ids),
            "shared_with_1.5x": len(shared_votu_ids),
            "jaccard_with_1.5x": (
                len(shared_votu_ids) / len(union_votu_ids)
                if union_votu_ids
                else np.nan
            ),
            "additional_relative_to_1.5x": len(
                retained_votu_ids.difference(default_votu_ids)
            ),
            "not_retained_relative_to_1.5x": len(
                default_votu_ids.difference(retained_votu_ids)
            ),
            "is_primary_threshold": key == DEFAULT_THRESHOLD_KEY,
        }
    )

threshold_summary = pd.DataFrame(threshold_summary_rows)
threshold_summary["threshold_numeric"] = pd.to_numeric(
    threshold_summary["threshold_multiplier"]
)
threshold_summary = threshold_summary.sort_values("threshold_numeric").drop(
    columns="threshold_numeric"
)

per_sample_background.to_csv(
    OUTDIR / "11_per_sample_microbial_background.csv",
    index=False,
)
per_sample_threshold_summary.to_csv(
    OUTDIR / "11_per_sample_threshold_summary.csv",
    index=False,
)
threshold_membership.to_csv(
    OUTDIR / "11_full_phyllosphere_threshold_membership.csv",
    index=False,
)
threshold_summary.to_csv(
    OUTDIR / "11_threshold_summary.csv",
    index=False,
)

threshold_summary_display = threshold_summary.copy()
threshold_summary_display["jaccard_with_1.5x"] = (
    threshold_summary_display["jaccard_with_1.5x"].round(3)
)
threshold_summary_display

print(
    "Saved the per-sample backgrounds, threshold membership, and "
    f"threshold summary in: {OUTDIR}"
)


Saved the per-sample backgrounds, threshold membership, and threshold summary in: /home/lmf/PhylloVir/VIRAL_WORLD/VIRAL_FRACTION/FIGURES_AND_TABLES/SUPPLEMENTARY/11_rank_abundance_threshold_selection


## Primary Threshold

The primary 1.5× rule requires a 50% margin above background.

In [ ]:
# Plot and export the primary-threshold selection
selection_figure_path = OUTDIR / "11_rank_abundance_threshold_selection.svg"

plot_summary = threshold_summary.copy()
plot_summary["threshold_numeric"] = pd.to_numeric(
    plot_summary["threshold_multiplier"]
)
primary_row = plot_summary.loc[
    plot_summary["is_primary_threshold"]
].iloc[0]

with sns.axes_style("ticks", {"axes.grid": True}), plt.rc_context(
    {
        "axes.linewidth": 0.5,
        "xtick.major.width": 0.5,
        "ytick.major.width": 0.5,
        "grid.linewidth": 0.25,
        "font.family": "sans-serif",
        "font.sans-serif": ["Liberation Sans"],
        "font.size": 6,
        "axes.labelsize": 7,
        "axes.labelweight": "bold",
        "xtick.labelsize": 6,
        "ytick.labelsize": 6,
        "svg.fonttype": "none",
    }
):
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(6.2, 2.45),
        gridspec_kw={"wspace": 0.38},
    )

    axes[0].plot(
        plot_summary["threshold_numeric"],
        plot_summary["n_retained_vOTUs"],
        color="#3399CC",
        marker="o",
        linewidth=1.1,
        markersize=3.5,
    )
    axes[0].scatter(
        primary_row["threshold_numeric"],
        primary_row["n_retained_vOTUs"],
        color="#F4C300",
        edgecolor="black",
        linewidth=0.45,
        s=30,
        zorder=3,
    )
    axes[0].annotate(
        "1.5x",
        (
            primary_row["threshold_numeric"],
            primary_row["n_retained_vOTUs"],
        ),
        xytext=(4, 6),
        textcoords="offset points",
        fontsize=6,
    )
    axes[0].set_xlabel("Background multiplier")
    axes[0].set_ylabel("Retained phyllosphere vOTUs (n)")

    axes[1].plot(
        plot_summary["threshold_numeric"],
        plot_summary["jaccard_with_1.5x"],
        color="#666666",
        marker="o",
        linewidth=1.1,
        markersize=3.5,
    )
    axes[1].scatter(
        primary_row["threshold_numeric"],
        primary_row["jaccard_with_1.5x"],
        color="#F4C300",
        edgecolor="black",
        linewidth=0.45,
        s=30,
        zorder=3,
    )
    axes[1].set_xlabel("Background multiplier")
    axes[1].set_ylabel("Jaccard overlap with 1.5x set")
    axes[1].set_ylim(-0.03, 1.03)

    for axis, panel_label in zip(axes, "AB"):
        axis.text(
            0.01, 1.02, panel_label, transform=axis.transAxes,
            fontsize=7, fontweight="bold", ha="left", va="bottom", zorder=10,
        )

    for axis in axes:
        axis.axvline(
            DEFAULT_THRESHOLD,
            color="black",
            linestyle="--",
            linewidth=0.65,
            zorder=0,
        )
        axis.set_xticks(THRESHOLD_MULTIPLIERS)
        axis.set_xticklabels([f"{value:g}x" for value in THRESHOLD_MULTIPLIERS])
        axis.spines[["top", "right"]].set_visible(False)

    fig.tight_layout()
    fig.savefig(selection_figure_path, format="svg", bbox_inches="tight")

plt.show()
plt.close(fig)


## Microbial-Reference Hold-Out

Estimate how often omitted eligible microbial references pass each multiplier.

In [5]:
# Estimate microbial-reference hold-out error
HOLDOUT_REPEATS = 1_000
HOLDOUT_RANDOM_SEED = 20260731
HOLDOUT_THRESHOLD_DEFINITIONS = [
    ("no_threshold", None),
    *[
        (threshold_key(multiplier), multiplier)
        for multiplier in THRESHOLD_MULTIPLIERS
    ],
]

# The held-out microbial reference pool is the exact unmasked 11 background pool after its existing
# length, plasmid, and predicted-phage exclusions. No masked references or
# additional filtering are introduced here.
holdout_background_tables = []
holdout_votu_tables = []

holdout_target_ids = sorted(phyllosphere_votu_ids)
holdout_target_index = {
    votu: index for index, votu in enumerate(holdout_target_ids)
}

for sample_index, sample in enumerate(SAMPLES):
    microbial_path = (
        VIRAL_MAPPING_REFERENCES_DIR
        / f"bowtie2_ALL_assembled_filtered_scaffolds_MICROBIAL_10000_derreplicated.tot_{sample}_tot_covstats.txt"
    )
    viral_path = VIRAL_MAPPING_DIR / f"bowtie2_{sample}_tot_covstats.txt"
    culture_path = (
        VIRAL_MAPPING_REFERENCES_DIR
        / f"bowtie2_strains_in_microbial_fraction_2018_{sample}_tot_covstats.txt"
    )
    mg_assembly_path = (
        VIRAL_MAPPING_REFERENCES_DIR
        / f"bowtie2_ALL_MG_assembled_filtered_scaffolds_MICROBIAL_10000_derreplicated.tot_{sample}_tot_covstats.txt"
    )

    tables = []
    for path, origin in [
        (microbial_path, "MICROBIAL"),
        (viral_path, "VIRAL"),
        (culture_path, "CULTURE"),
        (mg_assembly_path, "MG_ASSEMBLY"),
    ]:
        table = pd.read_csv(path, sep="\t")
        if table.shape[1] != len(covstats_columns):
            raise ValueError(
                f"Unexpected covstats columns for {sample}, {origin}: "
                f"{table.shape[1]}"
            )
        table.columns = covstats_columns
        table["ORIGIN"] = origin
        tables.append(table)

    combined = pd.concat(tables, ignore_index=True)
    combined["ID"] = combined["ID"].astype(str)
    combined["Trimmed_Mean"] = pd.to_numeric(
        combined["Trimmed_Mean"],
        errors="coerce",
    )
    combined["Length"] = pd.to_numeric(combined["Length"], errors="coerce")
    combined = combined.loc[combined["Trimmed_Mean"].gt(0)].copy()
    combined.loc[
        combined["ID"].isin(conservative_plasmid_ids),
        "ORIGIN",
    ] = "PLASMID"

    eligible_background = combined.loc[
        combined["ORIGIN"].isin(BACKGROUND_ORIGINS)
        & combined["Length"].ge(MIN_BACKGROUND_LENGTH)
        & ~combined["ID"].isin(predicted_phage_background_ids),
        ["ID", "ORIGIN", "Length", "Trimmed_Mean"],
    ].copy()
    eligible_background["background_key"] = (
        eligible_background["ORIGIN"] + "::" + eligible_background["ID"]
    )
    eligible_background["sample_index"] = sample_index
    holdout_background_tables.append(eligible_background)

    viral_rows = combined.loc[
        combined["ORIGIN"].eq("VIRAL")
        & combined["ID"].isin(holdout_target_index),
        ["ID", "Length", "Trimmed_Mean"],
    ].copy()
    viral_rows["sample_index"] = sample_index
    holdout_votu_tables.append(viral_rows)

holdout_background_long = pd.concat(
    holdout_background_tables,
    ignore_index=True,
)
holdout_background_long = (
    holdout_background_long
    .groupby(["background_key", "sample_index"], as_index=False)
    .agg(
        Trimmed_Mean=("Trimmed_Mean", "max"),
        Length=("Length", "first"),
    )
)
holdout_background_keys = sorted(
    holdout_background_long["background_key"].unique()
)
holdout_background_index = {
    key: index for index, key in enumerate(holdout_background_keys)
}
holdout_background_values = np.zeros(
    (len(holdout_background_keys), len(SAMPLES)),
    dtype=np.float32,
)
background_rows = holdout_background_long["background_key"].map(
    holdout_background_index
).to_numpy()
background_columns = holdout_background_long["sample_index"].to_numpy()
holdout_background_values[background_rows, background_columns] = (
    holdout_background_long["Trimmed_Mean"].to_numpy(dtype=np.float32)
)

holdout_votu_long = pd.concat(holdout_votu_tables, ignore_index=True)
holdout_votu_long = (
    holdout_votu_long
    .groupby(["ID", "sample_index"], as_index=False)
    .agg(
        Trimmed_Mean=("Trimmed_Mean", "max"),
        Length=("Length", "first"),
    )
)
holdout_votu_values = np.zeros(
    (len(holdout_target_ids), len(SAMPLES)),
    dtype=np.float32,
)
votu_rows = holdout_votu_long["ID"].map(holdout_target_index).to_numpy()
votu_columns = holdout_votu_long["sample_index"].to_numpy()
holdout_votu_values[votu_rows, votu_columns] = (
    holdout_votu_long["Trimmed_Mean"].to_numpy(dtype=np.float32)
)

full_background_values = holdout_background_values.max(axis=0)
full_votu_max_ratio = (
    holdout_votu_values / full_background_values
).max(axis=1)
observed_retained_counts = {
    "no_threshold": int((holdout_votu_values.max(axis=1) > 0).sum())
}
observed_retained_counts.update(
    {
        label: int((full_votu_max_ratio > multiplier).sum())
        for label, multiplier in HOLDOUT_THRESHOLD_DEFINITIONS
        if multiplier is not None
    }
)

# The reconstructed numeric counts must agree with 11 before any hold-out calculation
# estimates are calculated.
for label, multiplier in HOLDOUT_THRESHOLD_DEFINITIONS:
    if multiplier is None:
        continue
    expected_count = int(
        threshold_summary.loc[
            threshold_summary["threshold_multiplier"].eq(label),
            "n_retained_vOTUs",
        ].iloc[0]
    )
    if observed_retained_counts[label] != expected_count:
        raise ValueError(
            f"Could not reconstruct the 11 {label}x count: "
            f"{observed_retained_counts[label]} instead of {expected_count}."
        )

if len(holdout_background_keys) < len(holdout_target_ids):
    raise ValueError(
        "The eligible microbial-reference pool is smaller than the detected "
        "phyllosphere vOTU pool, so equal-sized panels of held-out microbial references cannot be drawn."
    )

holdout_input_summary = pd.DataFrame(
    [
        {
            "metric": "Detected phyllosphere vOTU candidates",
            "value": len(holdout_target_ids),
        },
        {
            "metric": "Eligible unmasked microbial references",
            "value": len(holdout_background_keys),
        },
        {
            "metric": "Microbial held-out microbial references per repeat",
            "value": len(holdout_target_ids),
        },
        {
            "metric": "Reference hold-out repeats",
            "value": HOLDOUT_REPEATS,
        },
    ]
)
holdout_input_summary.to_csv(
    OUTDIR / "11_microbial_holdout_input_summary.csv",
    index=False,
)
holdout_input_summary

# A held-out reference is removed from every sample before the top microbial
# background is recalculated. This models a bacterial sequence that was not
# represented in the background database.
background_rank_order = np.argsort(
    holdout_background_values,
    axis=0,
)[::-1]
rng = np.random.default_rng(HOLDOUT_RANDOM_SEED)
holdout_draw_rows = []

for repeat in range(HOLDOUT_REPEATS):
    held_out_reference_indices = rng.choice(
        len(holdout_background_keys),
        size=len(holdout_target_ids),
        replace=False,
    )
    held_out_mask = np.zeros(
        len(holdout_background_keys),
        dtype=bool,
    )
    held_out_mask[held_out_reference_indices] = True

    held_out_background = np.empty(len(SAMPLES), dtype=np.float32)
    for sample_index in range(len(SAMPLES)):
        for candidate_index in background_rank_order[:, sample_index]:
            if not held_out_mask[candidate_index]:
                held_out_background[sample_index] = (
                    holdout_background_values[candidate_index, sample_index]
                )
                break

    held_out_reference_values = holdout_background_values[held_out_reference_indices]
    held_out_reference_max_ratio = (held_out_reference_values / held_out_background).max(axis=1)
    held_out_votu_max_ratio = (
        holdout_votu_values / held_out_background
    ).max(axis=1)
    n_backgrounds_changed = int(
        (held_out_background != full_background_values).sum()
    )

    for label, multiplier in HOLDOUT_THRESHOLD_DEFINITIONS:
        if multiplier is None:
            held_out_reference_pass_mask = held_out_reference_values.max(axis=1) > 0
            votu_pass_mask = holdout_votu_values.max(axis=1) > 0
        else:
            held_out_reference_pass_mask = held_out_reference_max_ratio > multiplier
            votu_pass_mask = held_out_votu_max_ratio > multiplier

        n_held_out_references_passing = int(held_out_reference_pass_mask.sum())
        observed_votu_count = observed_retained_counts[label]
        holdout_draw_rows.append(
            {
                "repeat": repeat + 1,
                "threshold_multiplier": label,
                "observed_retained_vOTUs": observed_votu_count,
                "held_out_background_retained_vOTUs": int(votu_pass_mask.sum()),
                "n_held_out_references_passing": n_held_out_references_passing,
                "estimated_microbial_holdout_FDP_percent": (
                    100 * n_held_out_references_passing / observed_votu_count
                ),
                "n_sample_backgrounds_changed": n_backgrounds_changed,
            }
        )

microbial_holdout_draws = pd.DataFrame(holdout_draw_rows)
holdout_summary_rows = []

for label, multiplier in HOLDOUT_THRESHOLD_DEFINITIONS:
    threshold_draws = microbial_holdout_draws.loc[
        microbial_holdout_draws["threshold_multiplier"].eq(label)
    ]
    held_out_reference_counts = threshold_draws["n_held_out_references_passing"]
    fdp_values = threshold_draws[
        "estimated_microbial_holdout_FDP_percent"
    ]
    held_out_votu_counts = threshold_draws[
        "held_out_background_retained_vOTUs"
    ]

    holdout_summary_rows.append(
        {
            "threshold_multiplier": label,
            "observed_retained_vOTUs": observed_retained_counts[label],
            "n_held_out_references_per_repeat": len(holdout_target_ids),
            "n_repeats": HOLDOUT_REPEATS,
            "mean_held_out_references_passing": held_out_reference_counts.mean(),
            "median_held_out_references_passing": held_out_reference_counts.median(),
            "held_out_references_passing_2.5_percentile": held_out_reference_counts.quantile(0.025),
            "held_out_references_passing_97.5_percentile": held_out_reference_counts.quantile(0.975),
            "mean_estimated_microbial_holdout_FDP_percent": fdp_values.mean(),
            "median_estimated_microbial_holdout_FDP_percent": fdp_values.median(),
            "FDP_2.5_percentile": fdp_values.quantile(0.025),
            "FDP_97.5_percentile": fdp_values.quantile(0.975),
            "mean_held_out_background_retained_vOTUs": (
                held_out_votu_counts.mean()
            ),
            "mean_sample_backgrounds_changed": threshold_draws[
                "n_sample_backgrounds_changed"
            ].mean(),
        }
    )

microbial_holdout_summary = pd.DataFrame(holdout_summary_rows)
microbial_holdout_draws.to_csv(
    OUTDIR / "11_microbial_holdout_draws.csv",
    index=False,
)
microbial_holdout_summary.to_csv(
    OUTDIR / "11_microbial_holdout_summary.csv",
    index=False,
)

holdout_summary_display = microbial_holdout_summary.copy()
for column in holdout_summary_display.columns:
    if (
        "mean_" in column
        or "median_" in column
        or "percentile" in column
    ):
        holdout_summary_display[column] = (
            holdout_summary_display[column].round(3)
        )
holdout_summary_display


,threshold_multiplier,observed_retained_vOTUs,n_held_out_references_per_repeat,n_repeats,mean_held_out_references_passing,median_held_out_references_passing,held_out_references_passing_2.5_percentile,held_out_references_passing_97.5_percentile,mean_estimated_microbial_holdout_FDP_percent,median_estimated_microbial_holdout_FDP_percent,FDP_2.5_percentile,FDP_97.5_percentile,mean_held_out_background_retained_vOTUs,mean_sample_backgrounds_changed
0,no_threshold,7799,7799,1000,7799.000,7799.0,7799.0,7799.000,100.000,100.000,100.0,100.000,7799.000,4.155
1,1.0,1045,7799,1000,3.647,3.0,0.0,8.025,0.349,0.287,0.0,0.768,1105.122,4.155
2,1.25,861,7799,1000,1.773,2.0,0.0,5.000,0.206,0.232,0.0,0.581,910.838,4.155
3,1.5,726,7799,1000,1.131,1.0,0.0,4.000,0.156,0.138,0.0,0.551,769.928,4.155
4,2.0,559,7799,1000,0.688,0.0,0.0,3.000,0.123,0.000,0.0,0.537,592.445,4.155
5,3.0,397,7799,1000,0.267,0.0,0.0,2.000,0.067,0.000,0.0,0.504,420.757,4.155
6,5.0,254,7799,1000,0.151,0.0,0.0,1.000,0.059,0.000,0.0,0.394,268.723,4.155


## Hold-Out and PropagAtE-Style Support

Report hold-out risk and independent block-activity support separately.

In [6]:
# Pair hold-out estimates with PropagAtE-style support

BLOCK_ACTIVITY_LABELS_PATH = (
    VIRAL_FRACTION_DIR
    / "FIGURES_AND_TABLES"
    / "SUPPLEMENTARY"
    / "10_vOTU_block_activity"
    / "strain_depth_profiles"
    / "10_supplemented_votu_blocks_top0_rpkm_plus_votu_block_genomes_propagate_votu_labels.csv"
)

if not BLOCK_ACTIVITY_LABELS_PATH.exists():
    raise FileNotFoundError(
        "The supplemented-reference PropagAtE-style vOTU label table is missing: "
        f"{BLOCK_ACTIVITY_LABELS_PATH}"
    )

propagate_labels = pd.read_csv(
    BLOCK_ACTIVITY_LABELS_PATH,
    dtype={"vOTU": str},
)
required_propagate_columns = {
    "vOTU",
    "propagate_active",
    "n_samples_evaluated",
}
missing_propagate_columns = sorted(
    required_propagate_columns.difference(propagate_labels.columns)
)
if missing_propagate_columns:
    raise ValueError(
        "The supplemented-reference PropagAtE-style label table is missing columns: "
        f"{missing_propagate_columns}"
    )

propagate_labels["propagate_active"] = (
    propagate_labels["propagate_active"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(
        {
            "true": True,
            "t": True,
            "1": True,
            "yes": True,
            "false": False,
            "f": False,
            "0": False,
            "no": False,
        }
    )
)
if propagate_labels["propagate_active"].isna().any():
    raise ValueError(
        "Could not parse every supplemented-reference PropagAtE-style activity label."
    )
propagate_labels["n_samples_evaluated"] = pd.to_numeric(
    propagate_labels["n_samples_evaluated"],
    errors="raise",
)
propagate_labels = propagate_labels.loc[
    propagate_labels["n_samples_evaluated"].gt(0)
].copy()
if propagate_labels["vOTU"].duplicated().any():
    duplicate_votus = (
        propagate_labels.loc[
            propagate_labels["vOTU"].duplicated(),
            "vOTU",
        ]
        .head()
        .tolist()
    )
    raise ValueError(
        "Supplemented-reference PropagAtE-style labels are not one row per vOTU: "
        f"{duplicate_votus}"
    )
if propagate_labels.empty:
    raise ValueError("No evaluable supplemented-reference PropagAtE-style vOTU labels were found.")

def wilson_interval(successes, total, z_score=1.959963984540054):
    if total == 0:
        return np.nan, np.nan
    proportion = successes / total
    denominator = 1 + z_score**2 / total
    centre = (
        proportion + z_score**2 / (2 * total)
    ) / denominator
    half_width = (
        z_score
        * np.sqrt(
            proportion * (1 - proportion) / total
            + z_score**2 / (4 * total**2)
        )
        / denominator
    )
    return 100 * (centre - half_width), 100 * (centre + half_width)

propagate_concordance_rows = []

for label, multiplier in HOLDOUT_THRESHOLD_DEFINITIONS:
    if multiplier is None:
        retained_votu_ids = set(propagate_labels["vOTU"])
    else:
        retained_votu_ids = set(
            threshold_membership.loc[
                threshold_membership["threshold_multiplier"].eq(label)
                & threshold_membership["rank_abundance_call"].eq("retained"),
                "vOTU",
            ]
        )

    retained_mask = propagate_labels["vOTU"].isin(retained_votu_ids)
    active_mask = propagate_labels["propagate_active"]

    n_retained = int(retained_mask.sum())
    n_active_retained = int((retained_mask & active_mask).sum())
    n_inactive_retained = int((retained_mask & ~active_mask).sum())
    n_active_not_retained = int((~retained_mask & active_mask).sum())
    n_inactive_not_retained = int((~retained_mask & ~active_mask).sum())
    active_ci_low, active_ci_high = wilson_interval(
        n_active_retained,
        n_retained,
    )
    negative_ci_low, negative_ci_high = wilson_interval(
        n_inactive_retained,
        n_retained,
    )

    if multiplier is None or n_retained in {0, len(propagate_labels)}:
        odds_ratio = np.nan
        fisher_p_value = np.nan
    else:
        odds_ratio, fisher_p_value = fisher_exact(
            [
                [n_active_retained, n_inactive_retained],
                [n_active_not_retained, n_inactive_not_retained],
            ],
            alternative="two-sided",
        )

    fdp_row = microbial_holdout_summary.loc[
        microbial_holdout_summary["threshold_multiplier"].eq(label)
    ]
    if len(fdp_row) != 1:
        raise ValueError(
            f"Expected one microbial hold-out FDP row for {label}, "
            f"found {len(fdp_row)}."
        )
    fdp_row = fdp_row.iloc[0]

    propagate_concordance_rows.append(
        {
            "threshold_multiplier": label,
            "observed_retained_vOTUs": int(
                fdp_row["observed_retained_vOTUs"]
            ),
            "mean_microbial_holdout_FDP_percent": fdp_row[
                "mean_estimated_microbial_holdout_FDP_percent"
            ],
            "FDP_2.5_percentile": fdp_row["FDP_2.5_percentile"],
            "FDP_97.5_percentile": fdp_row["FDP_97.5_percentile"],
            "evaluable_vOTUs_total": len(propagate_labels),
            "evaluable_vOTUs_retained": n_retained,
            "PropagAtE_style_active": n_active_retained,
            "PropagAtE_style_not_active": n_inactive_retained,
            "PropagAtE_style_negative": n_inactive_retained,
            "PropagAtE_style_active_fraction_percent": (
                100 * n_active_retained / n_retained
                if n_retained
                else np.nan
            ),
            "PropagAtE_style_active_fraction_Wilson_95_low": active_ci_low,
            "PropagAtE_style_active_fraction_Wilson_95_high": active_ci_high,
            "PropagAtE_style_negative_fraction_percent": (
                100 * n_inactive_retained / n_retained
                if n_retained
                else np.nan
            ),
            "PropagAtE_style_negative_fraction_Wilson_95_low": negative_ci_low,
            "PropagAtE_style_negative_fraction_Wilson_95_high": negative_ci_high,
            "activity_odds_ratio_vs_not_retained": odds_ratio,
            "activity_Fisher_exact_P": fisher_p_value,
        }
    )

propagate_fdp_concordance = pd.DataFrame(propagate_concordance_rows)
propagate_fdp_concordance.to_csv(
    SUP_FIGS_TABLES_DIR / "11_microbial_holdout_FDP_PropagAtE_concordance.csv",
    index=False,

)
propagate_fdp_concordance.to_csv(
    SUP_FIGS_TABLES_DIR / "STable6_activity_threshold_validation.csv",
    index=False,
)

propagate_support_display_columns = [
    "threshold_multiplier",
    "evaluable_vOTUs_retained",
    "PropagAtE_style_active",
    "PropagAtE_style_negative",
    "PropagAtE_style_active_fraction_percent",
    "PropagAtE_style_negative_fraction_percent",
    "activity_odds_ratio_vs_not_retained",
    "activity_Fisher_exact_P",
]
propagate_support_display = propagate_fdp_concordance[
    propagate_support_display_columns
].copy()
for column in [
    "PropagAtE_style_active_fraction_percent",
    "PropagAtE_style_negative_fraction_percent",
    "activity_odds_ratio_vs_not_retained",
]:
    propagate_support_display[column] = (
        propagate_support_display[column].round(3)
    )
propagate_support_display["activity_Fisher_exact_P"] = (
    propagate_support_display["activity_Fisher_exact_P"].map(
        lambda value: (
            "NA"
            if pd.isna(value)
            else "<0.001"
            if value < 0.001
            else f"{value:.3f}"
        )
    )
)
propagate_support_display

primary_propagate_support = propagate_fdp_concordance.loc[
    propagate_fdp_concordance["threshold_multiplier"].eq(DEFAULT_THRESHOLD_KEY)
].iloc[0]
print(
    f"At {DEFAULT_THRESHOLD:g}x: "
    f"{int(primary_propagate_support['evaluable_vOTUs_retained'])} "
    "supplemented-reference blocks retained; "
    f"{int(primary_propagate_support['PropagAtE_style_active'])}/"
    f"{int(primary_propagate_support['evaluable_vOTUs_retained'])} "
    "PropagAtE-style active."
)


At 1.5x: 18 supplemented-reference blocks retained; 17/18 PropagAtE-style active.


## Independent Support Across Thresholds

A, microbial hold-out FDP; B, PropagAtE-style block support.

In [ ]:
# Plot independent support across thresholds
propagate_concordance_figure_path = (
    OUTDIR / "11_microbial_holdout_FDP_PropagAtE_concordance.svg"
)

concordance_plot = propagate_fdp_concordance.copy()
if "PropagAtE_style_negative_fraction_percent" not in concordance_plot.columns:
    concordance_plot["PropagAtE_style_negative_fraction_percent"] = (
        100 - concordance_plot["PropagAtE_style_active_fraction_percent"]
    )
    concordance_plot["PropagAtE_style_negative_fraction_Wilson_95_low"] = (
        100 - concordance_plot["PropagAtE_style_active_fraction_Wilson_95_high"]
    )
    concordance_plot["PropagAtE_style_negative_fraction_Wilson_95_high"] = (
        100 - concordance_plot["PropagAtE_style_active_fraction_Wilson_95_low"]
    )

all_x = np.arange(len(concordance_plot))
fdp_mean = concordance_plot[
    "mean_microbial_holdout_FDP_percent"
].to_numpy()
fdp_low = concordance_plot[
    "FDP_2.5_percentile"
].to_numpy()
fdp_high = concordance_plot[
    "FDP_97.5_percentile"
].to_numpy()
fdp_error = np.vstack(
    [
        fdp_mean - fdp_low,
        fdp_high - fdp_mean,
    ]
)
fdp_error = np.maximum(fdp_error, 0.0)

activity_fraction = concordance_plot[
    "PropagAtE_style_active_fraction_percent"
].to_numpy()
activity_low = concordance_plot[
    "PropagAtE_style_active_fraction_Wilson_95_low"
].to_numpy()
activity_high = concordance_plot[
    "PropagAtE_style_active_fraction_Wilson_95_high"
].to_numpy()
activity_error = np.vstack(
    [
        activity_fraction - activity_low,
        activity_high - activity_fraction,
    ]
)
activity_error = np.maximum(activity_error, 0.0)
negative_fraction = concordance_plot[
    "PropagAtE_style_negative_fraction_percent"
].to_numpy()
negative_low = concordance_plot[
    "PropagAtE_style_negative_fraction_Wilson_95_low"
].to_numpy()
negative_high = concordance_plot[
    "PropagAtE_style_negative_fraction_Wilson_95_high"
].to_numpy()
negative_error = np.vstack(
    [
        negative_fraction - negative_low,
        negative_high - negative_fraction,
    ]
)
negative_error = np.maximum(negative_error, 0.0)

all_labels = [
    "No\nthreshold",
    *[f"{multiplier:g}x" for multiplier in THRESHOLD_MULTIPLIERS],
]
primary_all_position = list(THRESHOLD_MULTIPLIERS).index(DEFAULT_THRESHOLD) + 1

with sns.axes_style("ticks", {"axes.grid": True}), plt.rc_context(
    {
        "axes.linewidth": 0.6,
        "xtick.major.width": 0.6,
        "ytick.major.width": 0.6,
        "grid.linewidth": 0.3,
        "font.family": "sans-serif",
        "font.sans-serif": ["Liberation Sans"],
        "font.size": 7,
        "axes.titlesize": 8,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "svg.fonttype": "none",
    }
):
    fig, axes = plt.subplots(
        nrows=3,
        ncols=1,
        figsize=(5.2, 5.2),
        sharex=True,
        constrained_layout=True,
        gridspec_kw={"height_ratios": [0.34, 0.82, 1.12]},
    )
    fdp_high_axis, fdp_low_axis, activity_axis = axes
    fdp_low_axis_upper = max(0.5, np.nanmax(fdp_high[1:]) * 1.18)

    for fdp_axis in [fdp_high_axis, fdp_low_axis]:
        fdp_axis.errorbar(
            all_x,
            fdp_mean,
            yerr=fdp_error,
            color="#3399CC",
            marker="o",
            markersize=3.7,
            linewidth=1.0,
            capsize=2,
            zorder=2,
        )
        fdp_axis.scatter(
            primary_all_position,
            fdp_mean[primary_all_position],
            color="#F4C300",
            edgecolor="black",
            linewidth=0.45,
            s=31,
            zorder=3,
        )
    fdp_high_axis.set_ylim(96, 104)
    fdp_high_axis.set_yticks([100])
    fdp_low_axis.set_ylim(-0.02, fdp_low_axis_upper)
    fdp_low_axis.set_ylabel("Microbial hold-out FDP (%)")
    fdp_high_axis.text(
        0.01, 1.02, "A", transform=fdp_high_axis.transAxes,
        fontsize=7, fontweight="bold", ha="left", va="bottom", zorder=10,
    )
    fdp_high_axis.spines["bottom"].set_visible(False)
    fdp_low_axis.spines["top"].set_visible(False)
    fdp_high_axis.tick_params(labelbottom=False, bottom=False)
    fdp_low_axis.tick_params(labelbottom=False)
    break_mark_kwargs = {
        "color": "black",
        "clip_on": False,
        "linewidth": 0.6,
    }
    break_mark_size = 0.012
    fdp_high_axis.plot(
        (-break_mark_size, break_mark_size),
        (-break_mark_size, break_mark_size),
        transform=fdp_high_axis.transAxes,
        **break_mark_kwargs,
    )
    fdp_high_axis.plot(
        (1 - break_mark_size, 1 + break_mark_size),
        (-break_mark_size, break_mark_size),
        transform=fdp_high_axis.transAxes,
        **break_mark_kwargs,
    )
    fdp_low_axis.plot(
        (-break_mark_size, break_mark_size),
        (1 - break_mark_size, 1 + break_mark_size),
        transform=fdp_low_axis.transAxes,
        **break_mark_kwargs,
    )
    fdp_low_axis.plot(
        (1 - break_mark_size, 1 + break_mark_size),
        (1 - break_mark_size, 1 + break_mark_size),
        transform=fdp_low_axis.transAxes,
        **break_mark_kwargs,
    )

    activity_axis.errorbar(
        all_x,
        activity_fraction,
        yerr=activity_error,
        color="#33CC99",
        marker="o",
        markersize=3.7,
        linewidth=1.0,
        capsize=2,
        zorder=2,
        label="Active",
    )
    activity_axis.errorbar(
        all_x,
        negative_fraction,
        yerr=negative_error,
        color="#CC6677",
        marker="s",
        markersize=3.4,
        linewidth=1.0,
        capsize=2,
        zorder=2,
        label="Negative",
    )
    activity_axis.scatter(
        primary_all_position,
        activity_fraction[primary_all_position],
        color="#F4C300",
        edgecolor="black",
        linewidth=0.45,
        s=31,
        zorder=3,
    )
    activity_axis.scatter(
        primary_all_position,
        negative_fraction[primary_all_position],
        color="#F4C300",
        edgecolor="black",
        marker="s",
        linewidth=0.45,
        s=28,
        zorder=3,
    )
    for x_position, row in concordance_plot.iterrows():
        activity_axis.text(
            x_position,
            103,
            f"n={int(row['evaluable_vOTUs_retained'])}",
            ha="center",
            va="bottom",
            fontsize=6,
            color="#444444",
        )
    activity_axis.set_xticks(all_x)
    activity_axis.set_xticklabels(all_labels)
    activity_axis.set_xlim(-0.35, len(all_x) - 0.65)
    activity_axis.set_ylim(-4, 112)
    activity_axis.set_xlabel("Background multiplier")
    activity_axis.set_ylabel("PropagAtE-style blocks (%)")
    activity_axis.text(
        0.01, 1.02, "B", transform=activity_axis.transAxes,
        fontsize=7, fontweight="bold", ha="left", va="bottom", zorder=10,
    )
    activity_axis.legend(
        frameon=False,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.28),
        ncol=2,
        handlelength=1.4,
        columnspacing=1.0,
        borderaxespad=0,
    )

    for axis in axes:
        axis.axvline(
            primary_all_position,
            color="black",
            linestyle="--",
            linewidth=0.55,
            zorder=0,
        )
        axis.spines[["top", "right"]].set_visible(False)

    fig.savefig(
        propagate_concordance_figure_path,
        format="svg",
        bbox_inches="tight",
    )
    fig.savefig(
        SUP_FIGS_TABLES_DIR / "SFigure2_threshold_validation_holdout_PropagAtE.svg",
        format="svg",
        bbox_inches="tight",
    )

plt.show()
plt.close(fig)
